In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.metrics.pairwise import cosine_similarity
import torch
from sentence_transformers import SentenceTransformer, models, losses, InputExample
from torch.utils.data import DataLoader

# --------- Step 1: Load your data ---------
df = pd.read_csv("../../testing.csv")
texts = df["combined"].astype(str).tolist()

# --------- Step 2: Define embedding function ---------
def embedding_model(texts, model_name):
    tokenizer = SentenceTransformer(model_name)
    embeddings = tokenizer.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    return embeddings

# --------- Step 3: Get pretrained embeddings ---------
model_name = "web3se/SmartBERT-v2"
embeddings = embedding_model(texts, model_name)

# --------- Step 4: Generate pseudo pairs ---------
sim_matrix = cosine_similarity(embeddings)

top_k = 3
positive_pairs = []
for idx, sims in enumerate(sim_matrix):
    # top_k similar excluding self
    similar_idx = sims.argsort()[::-1][1:top_k+1]
    for sim_idx in similar_idx:
        positive_pairs.append((texts[idx], texts[sim_idx], 1.0))

num_negatives = len(positive_pairs)
all_indices = list(range(len(texts)))
negative_pairs = []
for _ in range(num_negatives):
    i, j = random.sample(all_indices, 2)
    negative_pairs.append((texts[i], texts[j], 0.0))

all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)
train_examples = [InputExample(texts=[a,b], label=label) for a,b,label in all_pairs]

print(f"Generated {len(train_examples)} pairs: {len(positive_pairs)} positives and {len(negative_pairs)} negatives")

# --------- Step 5: Build SentenceTransformer model ---------
word_embedding_model = models.Transformer(model_name, max_seq_length=512)
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=False,
    pooling_mode_mean_tokens=True
)
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# --------- Step 6: Prepare DataLoader and Loss ---------
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

# --------- Step 7: Train ---------
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    output_path="./smartbert-contrastive"
)

print("Training finished and model saved to ./smartbert-contrastive")


C:\Users\Melis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?

In [3]:
!pip install --upgrade transformers


   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   --------- ------------------------------ 2.4/10.5 MB 11.2 MB/s eta 0:00:01
   ------------------ --------------------- 4.7/10.5 MB 11.4 MB/s eta 0:00:01
   ---------------------------- ----------- 7.3/10.5 MB 11.9 MB/s eta 0:00:01
   -------------------------------------- - 10.0/10.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 11.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.3
    Uninstalling transformers-4.52.3:
      Successfully uninstalled transformers-4.52.3
